# Real-stream preprocessing (mock-less galaxy 0000)

Sibling of `streams_preprocessing.ipynb`, but the **positive class is catalogued S5 members**, not injected mocks.

Composition: **DES background + real/S5 streams**. Injected mock stars are dropped. The same real members that were removed to train on mocks are put back. Cells, the frozen CMD slab, $\pi<1$ mas, the PM GMM, and the width cut are **the same label-free procedure** used for the mock graphs — knobs stay those fit on mock 0000; nothing is refit on S5.

Graph code still keys off `is_mock_stream`. After `prepare_real_table`, that column is **aliased to `is_real_stream`**, so occupancy plots and blob `y` mean “catalogued S5 member”. Plot titles that say “mock” in shared helpers mean S5 here.

```bash
bash scans/streams/submit_real_0000.sh
```

Output: `/scratch/gpfs/BHANIN/jgdezoort/streams/galaxies_real/train/0000` (mock training graphs are untouched). This notebook visualizes cut + GMM effects on named streams live from the parquet; GMM/blob panels load from disk when the pipeline has finished.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# 0. Load galaxy 0000, drop mocks, restore S5

Source parquet is the same DES+Gaia+mocks file the mock pipeline used. `load_galaxy` still applies the photometry gates ($g<22$, $r<20.5$). `prepare_real_table` then drops `is_mock` rows and keeps catalogued streams (ATLAS/Aliqa merged into AAU; the `Test` label is dropped as in the mock path).


In [ ]:
from cpen.apps.streams.preprocess.config import (
    BACKGROUND,
    CMD_SLAB_B,
    CMD_SLAB_HALF_WIDTH,
    CMD_SLAB_M,
    DEFAULT_GALAXY_ROOT,
    DEFAULT_REAL_PIPELINE_ROOT,
    PI_MAX,
    PSF_G_UPPER,
    PSF_R_UPPER,
)
from cpen.apps.streams.preprocess.des_footprint_centers import DES_FOOTPRINT_CENTERS
from cpen.apps.streams.preprocess.galaxy import load_galaxy, prepare_real_table, stream_cutflow_table
from cpen.apps.streams.preprocess.geometry import assign_cells
from cpen.apps.streams.preprocess.gmm import (
    GMM_N_COMPONENTS,
    GMM_WIDTH_MAX,
    load_blobs,
    load_gmm_cells,
    mock_stream_localization,
    occupancy_from_cells,
    print_gmm_localization,
    print_gmm_occupancy,
    print_width_cut_summary,
    summarize_width_cut,
)
from cpen.apps.streams.preprocess.physical import physical_keep_mask
from cpen.apps.streams.preprocess.plotting import (
    cmd_slab_intercepts,
    plot_aggregate_cmd,
    plot_aggregate_parallax_cmd,
    plot_cell_cmd,
    plot_cell_gmm,
    plot_cell_three_panel,
    plot_gmm_localization,
    plot_gmm_occupancy,
    plot_gmm_width_sweep,
    plot_named_streams_sky,
    plot_parallax_upper_bound_sweep,
    plot_stream_cutflow,
)
from cpen.apps.streams.preprocess.utils import print_mock_stream_cell_occupancy

GALAXY_PARQUET = Path(DEFAULT_GALAXY_ROOT) / "train" / "train_galaxy_0000_with_background.parquet"
REAL_OUT = Path(DEFAULT_REAL_PIPELINE_ROOT) / "train" / "0000"
CKPT = REAL_OUT / "checkpoint.parquet"

print(f"source   {GALAXY_PARQUET}")
print(f"real out {REAL_OUT}")
print(f"photometry gates  g < {PSF_G_UPPER}  r < {PSF_R_UPPER}")
print(f"frozen slab  m={CMD_SLAB_M}  b={CMD_SLAB_B}  half_w={CMD_SLAB_HALF_WIDTH}")
print(f"frozen π_max = {PI_MAX} mas   GMM k={GMM_N_COMPONENTS}  W={GMM_WIDTH_MAX} mas/yr")


In [ ]:
def cells_from_frame(frame: pd.DataFrame) -> dict:
    out = {}
    for col in frame.columns:
        if not str(col).startswith("in_cell_"):
            continue
        cid = int(str(col).split("_")[-1])
        sub = frame.loc[frame[col].to_numpy(bool)].copy()
        if len(sub):
            out[cid] = sub
    return out


def s5_host_cell_ids(frame: pd.DataFrame, k: int = 4) -> list[int]:
    m = frame["is_mock_stream"].to_numpy(bool)
    scores = []
    for col in frame.columns:
        if not str(col).startswith("in_cell_"):
            continue
        n = int((frame[col].to_numpy(bool) & m).sum())
        if n:
            scores.append((n, int(str(col).split("_")[-1])))
    scores.sort(reverse=True)
    return [cid for _, cid in scores[:k]]


if CKPT.is_file():
    print(f"Loading pipeline checkpoint {CKPT}")
    df = pd.read_parquet(CKPT)
    if "is_mock_stream" not in df.columns and "is_real_stream" in df.columns:
        df["is_mock_stream"] = df["is_real_stream"].to_numpy(bool)
else:
    print("No galaxies_real checkpoint yet — building the real-data table live.")
    df = load_galaxy(GALAXY_PARQUET)
    df = prepare_real_table(df, verbose=True)
    df, _ = assign_cells(df, centers=DES_FOOTPRINT_CENTERS, circle_radius_deg=5.0)

assert int(df["is_mock"].sum()) == 0, "expected zero injected mocks"
n_s5 = int(df["is_mock_stream"].sum())
print(f"N={len(df):,}  S5 members={n_s5:,}  (is_mock_stream aliased to is_real_stream)")

on_disk = load_gmm_cells(REAL_OUT, subdir="cells")
cell_dfs = on_disk if on_disk else cells_from_frame(df)
HOST_CELLS = s5_host_cell_ids(df, k=4)
print(f"S5-richest cells: {HOST_CELLS}  (cell tables from {'disk' if on_disk else 'in_cell flags'})")
display(
    df.loc[df["is_mock_stream"], "stream_label"]
    .astype(str)
    .value_counts()
    .rename("n_stars")
    .to_frame()
)


## 0.1 Named streams on the DES sky

Gray = field subsample. Color = catalogued members. Circles are the same 5° cells used for the mock graphs. Cells overlap, so a stream can sit in more than one circle.


In [ ]:
plot_named_streams_sky(df, centers=DES_FOOTPRINT_CENTERS, circle_radius=5.0)
_ = print_mock_stream_cell_occupancy(df, top_k=6, min_frac=0.05)


# 1. Physical cuts vs named streams

Same two label-free cuts as the mock pipeline: the **linear CMD slab** fit to 0000 *mocks* (not refit here) and **$\pi < 1$ mas** (missing Gaia parallax kept). The question is completeness of each S5 stream, not a new operating point.


In [ ]:
plot_aggregate_parallax_cmd(df)
if HOST_CELLS:
    plot_cell_cmd(cell_dfs, HOST_CELLS)


In [ ]:
b_lo, b_hi = cmd_slab_intercepts(CMD_SLAB_M, CMD_SLAB_B, CMD_SLAB_HALF_WIDTH)
plot_aggregate_cmd(df, slab=(CMD_SLAB_M, b_lo, b_hi), mark_mock_quantile=None)

g = df["psf_mag_aper_8_g_corrected_des"].to_numpy()
r = df["psf_mag_aper_8_r_corrected_des"].to_numpy()
is_s5 = df["is_mock_stream"].to_numpy(bool)
g_r = g - r
labs = df["stream_label"].astype(str).to_numpy()
names = [n for n in pd.Index(labs[is_s5]).value_counts().index]

fig, ax = plt.subplots(figsize=(7.2, 6.2), dpi=140)
ax.hexbin(g_r[~is_s5], r[~is_s5], gridsize=80, cmap="Greys", bins="log", mincnt=1, alpha=0.85)
xx = np.linspace(-0.4, 1.6, 200)
ax.plot(xx, CMD_SLAB_M * xx + b_lo, "k--", lw=1.0, label="frozen slab")
ax.plot(xx, CMD_SLAB_M * xx + b_hi, "k--", lw=1.0)
for i, name in enumerate(names):
    m = labs == name
    ax.scatter(g_r[m], r[m], s=12, alpha=0.85, label=f"{name} ({int(m.sum()):,})")
ax.set_xlim(-0.6, 2.0)
ax.set_ylim(22.5, 16.0)
ax.set_xlabel(r"$g-r$")
ax.set_ylabel(r"$r$")
ax.set_title("Frozen mock-0000 CMD slab vs catalogued streams")
ax.legend(fontsize=8, loc="lower left")
ax.grid(True, linestyle=":", alpha=0.4)
plt.show()


## 1.2 Parallax as a nearby-disk veto

Halo streams are distant. $\pi < 1$ mas is the same conservative Gaia foreground cut used for mocks. Completeness here is S5 members that survive, not a retune.


In [ ]:
plot_parallax_upper_bound_sweep(df, chosen=PI_MAX)


## 1.3 Sequential cutflow on named streams

Photometry is already applied by `load_galaxy`. Next: CMD slab, then $\pi<1$, then “in any DES cell”. Later cells add GMM assignment and $w \le W$ when those stages exist on disk.


In [ ]:
phys = physical_keep_mask(df)
g = df["psf_mag_aper_8_g_corrected_des"].to_numpy()
r = df["psf_mag_aper_8_r_corrected_des"].to_numpy()
g_r = g - r
from cpen.apps.streams.preprocess.physical import cmd_slab_signed_distance

cmd_ok = np.isfinite(g_r) & np.isfinite(r) & (
    np.abs(cmd_slab_signed_distance(g_r, r)) <= CMD_SLAB_HALF_WIDTH
)
plx = df["parallax_gaia"].to_numpy(float)
pi_ok = (~np.isfinite(plx)) | (plx < PI_MAX)
cell_cols = [c for c in df.columns if str(c).startswith("in_cell_")]
in_any = np.zeros(len(df), dtype=bool)
for c in cell_cols:
    in_any |= df[c].to_numpy(bool)

cutflow = stream_cutflow_table(
    df,
    [
        ("cmd_slab", cmd_ok),
        ("pi_lt_1", pi_ok),
        ("in_any_cell", in_any),
        ("physical", phys),
    ],
)
display(cutflow)
plot_stream_cutflow(cutflow, count_cols=["cmd_slab", "pi_lt_1", "in_any_cell"])


## 1.4 Example cells that actually host S5 members

Three-panel (sky / CMD / PM) for the DES cells with the most catalogued members. Highlighted points are S5 (via the `is_mock_stream` alias).


In [ ]:
PATCH = ("phi", "lam", "pm_phi", "pm_lam")
have_patch = bool(HOST_CELLS) and HOST_CELLS[0] in cell_dfs and all(
    c in cell_dfs[HOST_CELLS[0]].columns for c in PATCH
)
if not HOST_CELLS:
    print("No S5 members landed in a DES cell.")
elif have_patch:
    plot_cell_three_panel(cell_dfs, HOST_CELLS)
else:
    print("Patch-frame coords are on the per-cell parquets (stage 0). CMD only for now.")
    plot_cell_cmd(cell_dfs, HOST_CELLS)


# 2. GMM night-sky partition (same $k=64$ PM mixture)

Stage 2 tessellates each physically cut cell in $(\mu_\phi, \mu_\lambda)$. It is **not** a membership finder: every finite-PM star is assigned. Diagnostics below ask whether each named stream is concentrated in one (cell, Gaussian) or split across many — the same QC we used for mocks.

If `cells_gmm_k64/` is missing, submit `bash scans/streams/submit_real_0000.sh` and re-run from here.


In [ ]:
GMM_DIR = REAL_OUT / f"cells_gmm_k{GMM_N_COMPONENTS}"
PHYS_DIR = REAL_OUT / "cells_phys"
BLOB_DIR = REAL_OUT / "blobs"
GRAPH_DIR = REAL_OUT / "graphs"
S5_MIN = 5  # S5 hosts are smaller than mock-rich cells; QC floor of 30 is too high

gmm_cells = load_gmm_cells(REAL_OUT, subdir=f"cells_gmm_k{GMM_N_COMPONENTS}") or {}
if not gmm_cells:
    print(f"GMM cells not on disk yet: {GMM_DIR}")
    print("Submit:  bash scans/streams/submit_real_0000.sh")
    gmm_occ = None
    loc_sum = None
else:
    gmm_occ = occupancy_from_cells(gmm_cells)
    print_gmm_occupancy(gmm_occ, min_mock=S5_MIN)
    print_gmm_localization(gmm_occ, min_mock=S5_MIN)
    loc_sum, loc_chunks = mock_stream_localization(gmm_cells)
    print("\n===== Per named stream: GMM concentration =====")
    display(loc_sum)
    plot_gmm_occupancy(gmm_occ, min_mock=S5_MIN)
    plot_gmm_localization(gmm_occ, min_mock=S5_MIN)
    host_gmm = {cid: gmm_cells[cid] for cid in HOST_CELLS if cid in gmm_cells}
    if host_gmm:
        plot_cell_gmm(host_gmm, list(host_gmm))


## 2.2 Width cut $w \le 1.65$ mas/yr

Drop Gaussians that are too wide in PM. Aggregation is unique stars (a star in overlapping cells keeps the **minimum** width). Completeness is again S5, not a retune of $W$.


In [ ]:
if not gmm_cells:
    print("Skip width cut — GMM stage not on disk.")
else:
    from cpen.apps.streams.preprocess.gmm import sweep_gmm_width

    gmm_width_sweep, gmm_width_unique, gmm_width_ops = sweep_gmm_width(gmm_cells)
    display(gmm_width_ops)
    plot_gmm_width_sweep(gmm_width_sweep, chosen=GMM_WIDTH_MAX)
    print(f"Stage 3 GMM_WIDTH_MAX = {GMM_WIDTH_MAX:.3f} mas/yr (frozen from mocks)")
    stage3_tbl = summarize_width_cut(gmm_cells, width_max=GMM_WIDTH_MAX)
    print_width_cut_summary(stage3_tbl, min_mock=S5_MIN)

    parts = []
    for cid, sub in gmm_cells.items():
        t = sub.copy()
        t["_cid"] = cid
        parts.append(t)
    gmm_cat = pd.concat(parts, ignore_index=True)
    if "star_id" in gmm_cat.columns:
        gmm_cat = gmm_cat.sort_values("gmm_width").drop_duplicates("star_id", keep="first")
    w = gmm_cat["gmm_width"].to_numpy(float)
    w_ok = np.isfinite(w) & (w <= GMM_WIDTH_MAX)
    w_cutflow = stream_cutflow_table(gmm_cat, [("width_le_W", w_ok)])
    display(w_cutflow)
    plot_stream_cutflow(w_cutflow, count_cols=["width_le_W"])


# 3. Remaining blobs (graph units)

After $w \le W$, leftover Gaussians (including empty field blobs) are the graphs. Blobs with $N>8000$ are sky-split. **Keep every remaining blob** — S5 recovery is a later scoring pass on this same pool, not a cut that keeps only stream-rich chunks.


In [ ]:
blobs = load_blobs(REAL_OUT, subdir="blobs") if BLOB_DIR.is_dir() else {}
if not blobs:
    print(f"Blobs not on disk yet: {BLOB_DIR}")
else:
    rows = []
    for key, bdf in blobs.items():
        m = bdf["is_mock_stream"].to_numpy(bool) if "is_mock_stream" in bdf.columns else np.zeros(len(bdf), bool)
        n_pos = int(m.sum())
        host = None
        if n_pos and "stream_label" in bdf.columns:
            host = str(bdf.loc[m, "stream_label"].astype(str).value_counts().index[0])
        cid, gmm, part = (int(key[0]), int(key[1]), int(key[2]) if len(key) > 2 else 0)
        rows.append({
            "cell_id": cid,
            "gmm_label": gmm,
            "blob_part": part,
            "n_stars": len(bdf),
            "n_s5": n_pos,
            "host_stream": host,
        })
    blob_tab = pd.DataFrame(rows)
    print(f"{len(blob_tab):,} blobs  stars={int(blob_tab.n_stars.sum()):,}  S5={int(blob_tab.n_s5.sum()):,}")
    print(f"empty (n_s5=0): {(blob_tab.n_s5 == 0).mean():.1%}   "
          f"rich (n_s5>=100): {(blob_tab.n_s5 >= 100).sum()}")
    display(
        blob_tab.groupby("host_stream", dropna=False)
        .agg(n_blobs=("n_stars", "size"), n_stars=("n_stars", "sum"), n_s5=("n_s5", "sum"))
        .sort_values("n_s5", ascending=False)
    )
    rich = blob_tab.loc[blob_tab.n_s5 > 0].sort_values("n_s5", ascending=False).head(12)
    display(rich)


## 3.1 Scoring with a trained CPEN (next step)

Graphs under `galaxies_real/train/0000/graphs/` have node `y` = S5 membership. Point `BlobStreamDatamodule` / a Lightning `test` pass at that tree with a mock-trained `best.ckpt` to measure recovery. Do **not** train on these graphs — they are the held-out real sky.


In [ ]:
n_graphs = len(list(GRAPH_DIR.glob("*.npz"))) if GRAPH_DIR.is_dir() else 0
print(f"graphs on disk: {n_graphs:,}  ({GRAPH_DIR})")
print("Mock-trained checkpoint scoring on this tree is a follow-up, not this notebook.")
